## Chat history with memory

### Load ENV file

In [1]:
from dotenv import load_dotenv
import os

load = load_dotenv('./../.env', override=True)

print(os.getenv('LANGSMITH_PROJECT'))

EALangchainTrainingsTest


### Create local/cloud LLM object

In [4]:
from langchain_ollama import ChatOllama
import os

ollama_cloud_llm = ChatOllama(
    base_url="http://localhost:11434/",  # Ollama cloud endpoint
    model="gemini-3-flash-preview:cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
    temperature=0.5,
    max_tokens=350,
    headers={
        "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
    }
)

ollama_local_llm_1 = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest ",
    temperature=0.5,
    max_tokens=350
)

ollama_local_llm_2 = ChatOllama(
    base_url="http://localhost:11434/",
    model="qwen2.5:7b",
    temperature=0.5,
    max_tokens=350
)

### Message history with ChatMessageHistory

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [6]:
template = ChatPromptTemplate.from_messages([
    ("human", "{prompt}"),
    ("placeholder", "{history}")
])

chain = template | ollama_cloud_llm | StrOutputParser()

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="prompt",
    history_messages_key="history")

session_id = "Shashank"

get_session_history(session_id).clear

response1 = history.invoke(
    {"prompt": "What is the advantage of running the LLM in local machine?"},
    config={"configurable": {"session_id": session_id}})

response2 = history.invoke(
    {"prompt": "How about for cloud?"},
    config={"configurable": {"session_id": session_id}})

print(response1)
print("\n--------------------------------------------------------------------\n")
print(response2)

Running a Large Language Model (LLM) locally—on your own hardware rather than via a cloud provider like OpenAI or Google—offers several significant advantages. Depending on your use case (developer, privacy-conscious individual, or enterprise), these benefits can be game-changing.

Here are the primary advantages of running LLMs locally:

### 1. Data Privacy and Security
This is the most common reason for going local.
*   **No Data Leaks:** Your prompts, proprietary code, and sensitive documents never leave your machine. When you use cloud APIs, your data is sent to a third-party server, where it may be stored or used for further training.
*   **Compliance:** For industries like healthcare, finance, or law, running models locally makes it much easier to comply with regulations like HIPAA, GDPR, or SOC2, as the data remains within your controlled infrastructure.

### 2. Cost Efficiency (Long-Term)
While cloud APIs (like GPT-4) charge "per token," local models are essentially free to run